# Modelo Preditivo de Risco de Defasagem

Notebook da entrega do Datathon FIAP para construcao de um modelo que estima a probabilidade de um aluno entrar em risco de defasagem educacional.

## Etapas exigidas

- Carregamento e limpeza da base 2022-2024
- Feature engineering
- Separacao treino/teste
- Modelagem preditiva
- Avaliacao dos resultados
- Exportacao do modelo para uso no Streamlit

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.datathon_pipeline import DATA_PATH, MODEL_PATH, add_features, load_dataset, train_models

DATA_PATH

In [ ]:
data = load_dataset(DATA_PATH)
data.shape

In [ ]:
data[['Ano', 'IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV', 'IAN']].describe()

## Feature Engineering

O IAN foi tratado como indicador de adequação do nível: quanto menor o valor, maior a defasagem. Por isso, o alvo foi definido como `Risco_Defasagem = 1` quando `IAN < 7`.

Para evitar vazamento de informação, o IAN é usado apenas para criar o alvo de treino. Ele não entra como variável de entrada do modelo.

In [ ]:
model_data = add_features(data)
model_data[['Categoria_IAN', 'Risco_Defasagem']].value_counts().sort_index()

In [ ]:
model_data[
    ['IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV', 'IAN', 'Saude_Academica',
     'Bem_Estar_Psico', 'Risco_Composto', 'Gap_Expectativa_Realidade',
     'Coerencia_Autoavaliacao', 'Risco_Defasagem']
].head()

## Treino, Teste e Avaliacao

In [ ]:
bundle = train_models(model_data)
bundle['model_name'], bundle['target_definition']

In [ ]:
import pandas as pd

pd.DataFrame(bundle['metrics']).T[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

In [ ]:
import joblib

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, MODEL_PATH)
MODEL_PATH